# 1.1B - Persiapan Data: Lapisan Silver (Document Parsing & Structuring)

Notebook ini merupakan tahap kedua dalam pipeline **Retrieval-Augmented Generation (RAG)** menggunakan arsitektur **Medallion Lakehouse (Bronze -> Silver -> Gold)**.

### Ringkasan Proses:
1. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, dan konfigurasi dari `../Includes/Lab_Setup`.
2. **Membaca Tabel Lapisan Bronze**: Memuat data dokumen mentah (`bronze_docs_raw`) yang telah disiapkan pada notebook sebelumnya (`1.1A`).
3. **Eksplorasi Ekstraksi Dokumen dengan AI Functions (SQL)**: Menguji fungsi `ai_parse_document` untuk mengekstrak struktur dokumen biner, serta `ai_query` untuk mengubah JSON menjadi Markdown menggunakan model LLM.
4. **Transformasi Skala Penuh Menggunakan PySpark**: Menerapkan parsing dokumen dan konversi ke Markdown terstruktur secara batch ke seluruh data.
5. **Visualisasi Output Parsing**: Menginspeksi tata letak (*layout*) dan elemen dokumen hasil parse menggunakan modul helper `document_renderer`.
6. **Penyimpanan ke Tabel Lapisan Silver**: Menyimpan hasil ekstraksi dokumen ke tabel Delta terkelola `silver_docs_parsed` sebagai fondasi untuk tahap *chunking* dan pembuatan embedding pada lapisan Gold (`1.1C`).


### Langkah 1: Setup Lingkungan Lab
Menjalankan notebook setup untuk menginisialisasi variabel konfigurasi lab (seperti objek `course`, katalog, skema, dan volume di Unity Catalog).


In [0]:
%run ../Includes/Lab_Setup

### Langkah 2: Membaca Tabel Lapisan Bronze
Membaca tabel Delta `bronze_docs_raw` yang berisi dokumen biner mentah beserta metadata waktu penyerapan dari tahap 1.1A.


In [0]:
# Membaca tabel Delta lapisan Bronze yang berisi file dokumen mentah
bronze_df = spark.read.table("bronze_docs_raw")

# Menampilkan isi tabel untuk memverifikasi ketersediaan dan struktur data mentah
display(bronze_df)

### Langkah 3: Eksplorasi Parsing Dokumen Menggunakan AI Functions (SQL)
Menjalankan kueri SQL demonstrasi untuk melihat bagaimana Databricks AI Functions bekerja bersama:
- `ai_parse_document`: Mengekstrak struktur dokumen mentah (halaman, teks, tabel, gambar) ke format JSON.
- `ai_query`: Memanfaatkan model LLM (`databricks-gpt-oss-20b`) untuk menyusun kembali JSON hasil parse menjadi teks Markdown yang bersih dan mudah dibaca dengan pemisah halaman `== page ==`.


In [0]:
%sql
-- Common Table Expression (CTE) untuk mem-parsing dokumen mentah menjadi representasi JSON terstruktur
WITH parsed_docs AS (
  SELECT path,
        -- Menggunakan AI Function bawaan Databricks untuk mengekstrak konten dan layout dokumen
        ai_parse_document(content, map('version', '2.0')) as parsed_content
  FROM bronze_docs_raw
)
-- Menggunakan model LLM melalui `ai_query` untuk mengonversi JSON dokumen menjadi Markdown yang rapi
SELECT path,
      parsed_content,
      ai_query('databricks-gpt-oss-20b',
               CONCAT('Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page without adding page number. Preserve important structure such as pages, headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.
               JSON:', CAST(parsed_content AS STRING))
              )
FROM parsed_docs

### Langkah 4: Pemrosesan Dokumen Skala Penuh Menggunakan PySpark
Menerapkan ekstraksi dokumen dan konversi ke Markdown secara terdistribusi pada DataFrame PySpark:
1. `parsed_content`: Mem-parsing dokumen biner dan menyimpan gambar yang diekstrak ke dalam Unity Catalog Volume (`{course.dataset_volume}/imgs`).
2. `plain_text`: Mengonversi JSON parsed content menjadi format teks Markdown menggunakan LLM endpoint `databricks-gpt-oss-20b`.


In [0]:
from pyspark.sql.functions import expr

# Endpoint model LLM Databricks Foundation Model API yang digunakan untuk formatting teks
endpoint = "databricks-gpt-oss-20b"

# Prompt instruksi untuk LLM agar menghasilkan Markdown bersih dengan pemisah '== page =='
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page without adding page number. Preserve important structure such as pages, headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.

JSON:

'''

# Memproses data mentah menjadi data terstruktur (Lapisan Silver)
parsed_df = (bronze_df.withColumn("parsed_content",
                                  # ai_parse_document: mengekstrak elemen dokumen dan menyimpan gambar hasil ekstraksi ke folder /imgs
                                  expr(f"""ai_parse_document(content, map("version", "2.0",
                                                                          "imageOutputPath", "{course.dataset_volume}/imgs")
                                                            )"""))
                      .withColumn("plain_text",
                                  # ai_query: memanggil LLM untuk merangkum dan menyusun JSON menjadi dokumen Markdown terstruktur
                                  expr(f"""ai_query('{endpoint}',
                                                    CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING)),
                                                    responseFormat => '{{"type":"text"}}'
                                                  )"""))
                      # Memilih kolom yang relevan untuk kebutuhan downstream RAG
                      .select("path", "parsed_content", "plain_text")
            )

# Menampilkan DataFrame hasil parsing dan konversi Markdown
display(parsed_df)

### Langkah 5: Visualisasi dan Inspeksi Hasil Parsing Dokumen
Menggunakan modul pembantu (*helper module*) `document_renderer` untuk menginspeksi hasil ekstraksi dokumen (halaman, kotak pembatas/bounding box, teks, dan tabel) secara visual.


In [0]:
import sys, os

# Menambahkan folder Includes ke sys.path agar modul kustom dapat diimpor
sys.path.append(os.path.abspath('../Includes'))
from document_renderer import render_ai_parse_output, render_ai_parse_output_interactive

# Mengambil 1 sampel hasil parsing dokumen untuk inspeksi visual
sample = parsed_df.select("parsed_content").limit(1).collect()
doc = sample[0]["parsed_content"]

# Merender tampilan visual hasil ekstraksi dokumen (halaman, teks, dan elemen layout)
render_ai_parse_output(doc)

### Langkah 6: Menyimpan Data ke Tabel Delta Lapisan Silver
Menyimpan DataFrame hasil ekstraksi dan formatting dokumen ke tabel Delta terkelola `silver_docs_parsed`.
Tabel lapisan Silver ini menyediakan teks yang sudah bersih (`plain_text`) dan metadata dokumen (`parsed_content`), yang akan digunakan pada tahap **Gold (1.1C)** untuk pemotongan teks (*chunking*) dan pembuatan representasi vektor (*embedding*).


In [0]:
# Menyimpan hasil pemrosesan ke tabel Delta Lake dengan mode overwrite
# agar data pada lapisan Silver selalu sinkron dan terkelola (managed table)
parsed_df.write.mode("overwrite").saveAsTable("silver_docs_parsed")